## Does the model beat the market?

For each game from 2015–2025 (out-of-sample), we check whether the model's
side against the closing spread was right. Break-even at -110 odds is 52.4%.

| Minimum disagreement | Games | Model correct |
|---|---|---|
| Any | 3,028 | 51.9% |
| 2+ points | 1,357 | 52.3% |
| 5+ points | 219 | 49.8% |

The model is slightly better than a coin flip but below break-even, and its
largest disagreements were its least accurate, usually because the market
knew something the model didn't (often a QB change).

*Model evaluation, not betting advice.*

In [ ]:
import numpy as np
import pandas as pd
from nflmodel.features.build import build_features
from nflmodel.models.hybrid import HybridModel
from nflmodel.config import BACKTEST_SEASONS

games = build_features()
rows = []
for season in BACKTEST_SEASONS:
    train = games[games["season"] < season]
    test = games[games["season"] == season].copy()
    test["pred"] = HybridModel().fit(train).predict(test)
    rows.append(test)
preds = pd.concat(rows)

edge = preds["pred"] - preds["spread_line"]       # + means we like the home team more than Vegas
cover = preds["result"] - preds["spread_line"]    # + means the home team covered

for threshold in [0, 2, 3, 4, 5]:
    bets = preds[(edge.abs() >= threshold) & (edge != 0)]
    e, c = edge[bets.index], cover[bets.index]
    won = np.sign(e) == np.sign(c)
    push = c == 0
    profit = np.where(push, 0, np.where(won, 0.909, -1.0))
    decided = (~push).sum()
    print(f"edge >= {threshold}: {len(bets):4d} bets, "
          f"win rate {won[~push].mean():.1%}, "
          f"profit {profit.sum():+.1f} units, ROI {profit.sum() / len(bets):+.1%}")